# 用 Deepgram 转写音频文件，并用 Anthropic 准备面试问题！

**把这份笔记本复制一份到你自己的云端硬盘，然后按下面的说明操作！** 🥳🥳🥳

----------------------------

# 开始使用：
依次运行下面三个单元格，你就能转写任意你想转写的音频。下方的注释指出了你可以调整的变量，用来按需修改输出。

在运行这份笔记本之前，你需要准备几个待转写的音频 URL。你可以使用任意音频文件。

顺便说一下，如果你还没有注册 Deepgram，请看看这个链接：https://dpgr.am/prerecorded-notebook-signup

# 第 1 步：依赖

运行此单元格以下载所有必需的依赖。

说明：你可以点击单元格左侧的播放按钮来运行它，也可以点击该单元格后同时按 `shift`+`ENTER`。（在 Mac 上是 `shift` + `return`。）

In [ ]:
%pip install requests ffmpeg-python
%pip install deepgram-sdk --upgrade
%pip install requests
%pip install anthropic

# 第 2 步：音频 URL 文件

找一些托管在服务器上的音频文件，以便使用这份笔记本。或者：下面的代码提供了 Deepgram 提供的一个示例文件。

In [ ]:
# Have you completed Step 2 above? 👀
# Do you see your audio file in the folder on the left? 📂

# 第 3 步：转写

填写以下变量：


* `DG_KEY` = 你个人的 Deepgram API 密钥
* `AUDIO_FILE_URL` = 你想转写的音频文件的 URL。


现在运行该单元格！（`Shift` + `Enter`）

-----------



顺便说一下，如果你已经是 Deepgram 用户，而该单元格报错，最常见的解决办法是：

1. 你可能需要更新已安装的 deepgram-sdk。
2. 你可能需要检查 Deepgram 账户中还剩多少额度。

In [ ]:
import requests
from deepgram import DeepgramClient, FileSource, PrerecordedOptions

# Deepgram API key
DG_KEY = "🔑🔑🔑 Your API Key here! 🔑🔑🔑"

# URL of the audio file
AUDIO_FILE_URL = "https://static.deepgram.com/examples/nasa-spacewalk-interview.wav"

# Path to save the transcript JSON file
TRANSCRIPT_FILE = "transcript.json"


def main():
    try:
        # STEP 1: Create a Deepgram client using the API key
        deepgram = DeepgramClient(DG_KEY)

        # Download the audio file from the URL
        response = requests.get(AUDIO_FILE_URL, timeout=60)
        if response.status_code == 200:
            buffer_data = response.content
        else:
            print("Failed to download audio file")
            return

        payload: FileSource = {
            "buffer": buffer_data,
        }

        # STEP 2: Configure Deepgram options for audio analysis
        options = PrerecordedOptions(
            model="nova-2",
            smart_format=True,
        )

        # STEP 3: Call the transcribe_file method with the text payload and options
        response = deepgram.listen.prerecorded.v("1").transcribe_file(payload, options)

        # STEP 4: Write the response JSON to a file
        with open(TRANSCRIPT_FILE, "w") as transcript_file:
            transcript_file.write(response.to_json(indent=4))

        print("Transcript JSON file generated successfully.")

    except Exception as e:
        print(f"Exception: {e}")


if __name__ == "__main__":
    main()

如果上面的单元格执行成功，你应该能在 content 目录中看到 JSON 输出文件。说明：单元格运行结束与 JSON 文件真正出现之间可能有一点时延，这是正常现象。稍等片刻，等待文件出现即可。

# 第 4 步：查看你的转写结果

下面的函数解析输出 JSON，并打印你刚刚转写的某个文件的转写文本！（请确认你要查看的文件确实已经加载到 content 目录中。）

**把 `OUTPUT` 变量设置为你想查看转写文本的文件名。**

然后运行此单元格（`Shift`+`Enter`），即可逐句查看你音频的转写结果！

In [ ]:
import json

# Set this variable to the path of the output file you wish to read
OUTPUT = "transcript.json"


# The JSON is loaded with information, but if you just want to read the
# transcript, run the code below!
def print_transcript(transcription_file):
    with open(transcription_file) as file:
        data = json.load(file)
        result = data["results"]["channels"][0]["alternatives"][0]["transcript"]
        result = result.split(".")
        for sentence in result:
            print(sentence + ".")


print_transcript(OUTPUT)


如果上面的单元格执行成功，你应该能看到音频转写的纯文本版本。

# 第 5 步：使用 Anthropic 准备面试问题

现在我们可以把转写文本发送给 Anthropic 进行分析，帮助我们准备一些面试问题。运行下面的单元格（`Shift`+`Enter`），即可获得 Anthropic 基于你上面音频转写文本给出的面试问题建议。

In [ ]:
import json

import anthropic

transcription_file = "transcript.json"


# Function to get the transcript from the JSON file
def get_transcript(transcription_file):
    with open(transcription_file) as file:
        data = json.load(file)
        result = data["results"]["channels"][0]["alternatives"][0]["transcript"]
        return result


# Load the transcript from the JSON file
message_text = get_transcript(transcription_file)

# Initialize the Claude API client
client = anthropic.Anthropic(
    # Defaults to os.environ.get("ANTHROPIC_API_KEY")
    # Claude API key
    api_key="🔑🔑🔑 Your API Key here! 🔑🔑🔑"
)

# Prepare the text for the API request
formatted_messages = [{"role": "user", "content": message_text}]

# Generate thoughtful, open-ended interview questions
response = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=1000,
    temperature=0.5,
    system="Your task is to generate a series of thoughtful, open-ended questions for an interview based on the given context. The questions should be designed to elicit insightful and detailed responses from the interviewee, allowing them to showcase their knowledge, experience, and critical thinking skills. Avoid yes/no questions or those with obvious answers. Instead, focus on questions that encourage reflection, self-assessment, and the sharing of specific examples or anecdotes.",
    messages=formatted_messages,
)

# Print the generated questions

# Join the text of each TextBlock into a single string
content = "".join(block.text for block in response.content)

# Split the content by '\n\n'
parts = content.split("\n\n")

# Print each part with an additional line break
for part in parts:
    print(part)
    print("\n")

如果该单元格执行成功，你应该能看到一份基于原始音频文件的面试问题列表。现在你已经可以用 Deepgram 转写音频，并用 Anthropic 得到一组面试问题。